In [0]:

from pyspark.sql import functions as F

# PHASE 2 — Majority-class baseline

snapshot_root = (
    "abfss://curated@stflight60106354.dfs.core.windows.net/"
    "flight_delays/phase2/phase2_20251118_v1/"
)

train = spark.read.parquet(snapshot_root + "train/")
validation = spark.read.parquet(snapshot_root + "validation/")

# Learn the most common target using TRAINING data only.
class_counts = (
    train.groupBy("ArrDel15")
    .count()
    .orderBy(F.desc("count"), F.asc("ArrDel15"))
    .collect()
)

majority_class = int(class_counts[0]["ArrDel15"])
print("Baseline predicted class:", majority_class)

# Predict the majority class on validation records.
predictions = validation.select(
    F.col("ArrDel15").cast("int").alias("label")
).withColumn(
    "prediction", F.lit(majority_class)
)

metrics = predictions.agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(F.col("label") == F.col("prediction"), 1)
         .otherwise(0)
    ).alias("correct"),
    F.sum(F.when(F.col("label") == 1, 1).otherwise(0)).alias("positive"),
    F.sum(F.when(F.col("label") == 0, 1).otherwise(0)).alias("negative"),
    F.sum(
        F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1)
         .otherwise(0)
    ).alias("tp"),
    F.sum(
        F.when((F.col("label") == 0) & (F.col("prediction") == 0), 1)
         .otherwise(0)
    ).alias("tn")
).first()

accuracy = metrics["correct"] / metrics["total"]
recall_delayed = metrics["tp"] / metrics["positive"]
recall_on_time = metrics["tn"] / metrics["negative"]
balanced_accuracy = (recall_delayed + recall_on_time) / 2

print("\nBASELINE VALIDATION RESULTS")
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_accuracy:.4f}")
print(f"Delayed-flight recall: {recall_delayed:.4f}")
print("Validation records:", metrics["total"])
print("PASS: Baseline evaluation completed")


Baseline predicted class: 0

BASELINE VALIDATION RESULTS
Accuracy: 0.8145
Balanced accuracy: 0.5000
Delayed-flight recall: 0.0000
Validation records: 1159898
PASS: Baseline evaluation completed
